# 02 · Preparación de datos y modelos supervisados

EP2 · Machine Learning · Waymo Open Dataset v2.0.1 (`lidar_box`)

Este notebook tiene dos partes. La **Parte A** (este paso) deja los datos listos y define cómo vamos a evaluar sin hacer trampa. La **Parte B** (siguiente paso) entrena y compara los modelos.

Las dos tareas:
- **Clasificación:** predecir el tipo de objeto (Vehículo, Señal, Peatón, Ciclista).
- **Regresión:** predecir la distancia entre el vehículo y el objeto, en metros.

In [ ]:
import sys
sys.path.append("..")  # permite importar el paquete src/ desde notebooks/

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold, cross_validate
from sklearn.pipeline import Pipeline

from src.data_preprocessing import (
    FEATURES_CLASIFICACION,
    FEATURES_REGRESION,
    RANDOM_STATE,
    agregar_features,
    agregar_nombre_tipo,
    cargar_segmentos,
    construir_preprocesador,
    dividir_por_segmentos,
    elegir_segmentos_prueba,
    preparar_clasificacion,
    preparar_regresion,
    submuestrear_frames,
    verificar_sin_fuga,
)

ORDEN_CLASES = ["Vehículo", "Señal", "Peatón", "Ciclista"]

## Parte A · Preparación de los datos

### 1. Carga y variables derivadas

Además de las columnas originales, calculamos cuatro variables que resumen mejor lo que mide el sensor:
- `volumen`: largo × ancho × alto de la caja.
- `rapidez`: velocidad total del objeto (m/s), sin importar la dirección.
- `aceleracion`: aceleración total del objeto.
- `distancia_vehiculo`: distancia en metros entre el vehículo y el objeto.

In [ ]:
df = agregar_features(agregar_nombre_tipo(cargar_segmentos("../data/raw/lidar_box")))
print(f"Filas: {len(df):,} | Segmentos: {df['key_segment_context_name'].nunique()}")
df[["volumen", "rapidez", "aceleracion", "distancia_vehiculo"]].describe().round(2)

### 2. Reducir frames repetidos

El sensor mide 10 veces por segundo, así que un mismo objeto aparece casi igual en frames seguidos. Nos quedamos con **1 frame de cada 5** dentro de cada segmento. Así el entrenamiento es unas 5 veces más rápido y casi no se pierde información (la pauta pide usar bien los recursos de cómputo). Revisamos que no se pierdan objetos, sobre todo ciclistas.

In [ ]:
CADA_N = 5
df_sub = submuestrear_frames(df, cada_n=CADA_N)

comparacion = pd.DataFrame({
    "filas_antes": df["type_nombre"].value_counts().reindex(ORDEN_CLASES),
    "filas_despues": df_sub["type_nombre"].value_counts().reindex(ORDEN_CLASES),
    "objetos_antes": df.groupby("type_nombre")["key_laser_object_id"].nunique().reindex(ORDEN_CLASES),
    "objetos_despues": df_sub.groupby("type_nombre")["key_laser_object_id"].nunique().reindex(ORDEN_CLASES),
})
print(f"Filas: {len(df):,} -> {len(df_sub):,}")
comparacion

**Hallazgo:** _(¿se perdió algún objeto de alguna clase? ¿cuántos ciclistas quedan?)._

### 3. Separar entrenamiento y prueba por segmentos completos

Reservamos cerca del 20 % de los **segmentos** para una prueba final que no se toca hasta el final. Los segmentos se separan completos (nunca se parten), así el modelo no se prueba con objetos que ya vio. La elección usa `StratifiedGroupKFold` para que las clases queden repartidas de forma parecida, y una semilla fija para que sea reproducible.

In [ ]:
segmentos_prueba = elegir_segmentos_prueba(df, random_state=RANDOM_STATE)
df_train, df_test = dividir_por_segmentos(df_sub, segmentos_prueba)
verificar_sin_fuga(df_train, df_test)  # falla si algún segmento u objeto está en los dos lados
print("Sin fuga entre entrenamiento y prueba: OK")

def resumen(d):
    return pd.Series({
        "segmentos": d["key_segment_context_name"].nunique(),
        "filas": len(d),
        "objetos": d["key_laser_object_id"].nunique(),
        **{f"objetos {c}": d.loc[d["type_nombre"] == c, "key_laser_object_id"].nunique() for c in ORDEN_CLASES},
    })

pd.DataFrame({"entrenamiento": resumen(df_train), "prueba": resumen(df_test)})

**Hallazgo:** _(cuántos segmentos y cuántos ciclistas quedaron en la prueba; ¿qué tan confiable será medir ciclistas ahí?)._

### 4. Variables de entrada de cada tarea

**Importante:** en la regresión **no** usamos `box_center_x/y/z` ni `distancia_vehiculo`, porque la distancia se calcula directamente con esas columnas. Eso sería hacer trampa. El modelo tiene que estimar la distancia solo con lo que mide el LiDAR sobre el objeto: su tipo, su tamaño y cuántos puntos recibió.

In [ ]:
print("Clasificación:", FEATURES_CLASIFICACION)
print("Regresión:     ", FEATURES_REGRESION)

X_clf_train, y_clf_train, g_clf_train = preparar_clasificacion(df_train)
X_reg_train, y_reg_train, g_reg_train = preparar_regresion(df_train)
print(f"\nEntrenamiento -> clasificación: {X_clf_train.shape} | regresión: {X_reg_train.shape}")

### 5. Transformación de variables dentro de un pipeline

Las variables se transforman dentro de un `Pipeline` de Scikit-learn:
- Los conteos de puntos LiDAR son muy desiguales (de 0 a miles), así que se les aplica `log(1 + x)`.
- Todas las variables numéricas se escalan (media 0, desviación 1), lo que necesitan los modelos lineales y KNN.
- En regresión, el tipo de objeto se convierte en columnas 0/1 (One-Hot).

Al ir dentro del pipeline, el escalado se calcula **solo con los datos de entrenamiento de cada fold**, así no se filtra información de los datos de validación.

In [ ]:
prep_clf = construir_preprocesador("clasificacion").fit(X_clf_train)
prep_reg = construir_preprocesador("regresion").fit(X_reg_train)
print("Clasificación ->", list(prep_clf.get_feature_names_out()))
print("Regresión     ->", list(prep_reg.get_feature_names_out()))

### 6. Validación cruzada agrupada por segmento

Con los datos de entrenamiento se hace validación cruzada de 5 folds, **sin partir ningún segmento**:
- **Clasificación:** `StratifiedGroupKFold`, que además intenta repartir las clases parejo.
- **Regresión:** `GroupKFold`.

Veamos qué le toca a cada fold, sobre todo en la clase Ciclista.

In [ ]:
cv_clf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_reg = GroupKFold(n_splits=5)

filas = []
for k, (_, idx_val) in enumerate(cv_clf.split(X_clf_train, y_clf_train, g_clf_train), start=1):
    val = y_clf_train.iloc[idx_val]
    filas.append({"fold": k,
                  "segmentos": g_clf_train.iloc[idx_val].nunique(),
                  "filas": len(idx_val),
                  **{c: int((val == c).sum()) for c in ORDEN_CLASES}})
print("Clasificación: filas de validación por fold y clase")
display(pd.DataFrame(filas).set_index("fold"))

filas = []
for k, (_, idx_val) in enumerate(cv_reg.split(X_reg_train, y_reg_train, g_reg_train), start=1):
    filas.append({"fold": k, "segmentos": g_reg_train.iloc[idx_val].nunique(), "filas": len(idx_val)})
print("Regresión: filas de validación por fold")
display(pd.DataFrame(filas).set_index("fold"))

**Hallazgo:** _(¿algún fold se queda sin ciclistas o con muy pocos? ¿qué implica para el recall de esa clase?)._

### 7. Línea base: qué resultado se obtiene sin aprender nada

Antes de entrenar modelos, medimos modelos "tontos". Son el piso que cualquier modelo real debe superar:
- Clasificación: siempre responder la clase más frecuente.
- Regresión: siempre responder la distancia promedio.

In [ ]:
def resumir(res, metricas):
    tabla = pd.DataFrame({m: [res[f"test_{m}"].mean(), res[f"test_{m}"].std()] for m in metricas},
                         index=["media", "desv. entre folds"])
    return tabla.round(3)

base_clf = Pipeline([("prep", construir_preprocesador("clasificacion")),
                     ("modelo", DummyClassifier(strategy="most_frequent"))])
res_clf = cross_validate(base_clf, X_clf_train, y_clf_train, groups=g_clf_train, cv=cv_clf,
                         scoring=["accuracy", "balanced_accuracy", "f1_macro"])
print("Línea base de clasificación (siempre la clase más frecuente)")
display(resumir(res_clf, ["accuracy", "balanced_accuracy", "f1_macro"]))

base_reg = Pipeline([("prep", construir_preprocesador("regresion")),
                     ("modelo", DummyRegressor(strategy="mean"))])
res_reg = cross_validate(base_reg, X_reg_train, y_reg_train, groups=g_reg_train, cv=cv_reg,
                         scoring=["r2", "neg_root_mean_squared_error", "neg_mean_absolute_error"])
res_reg["test_rmse"] = -res_reg["test_neg_root_mean_squared_error"]
res_reg["test_mae"] = -res_reg["test_neg_mean_absolute_error"]
print("Línea base de regresión (siempre la distancia promedio)")
display(resumir(res_reg, ["r2", "rmse", "mae"]))

**Hallazgo:** _(¿qué accuracy logra el modelo que siempre dice la clase más frecuente y qué dice su balanced accuracy? ¿por qué sirve tener las dos métricas?)._

## Decisiones de la Parte A

_(complétalo con tus palabras: variables elegidas, por qué no se usan las coordenadas en regresión, submuestreo de frames, división por segmentos y la línea base que hay que superar)._

## Parte B · Entrenamiento y comparación de modelos

Ahora entrenamos varios modelos para cada tarea y los comparamos con **la misma validación cruzada** (los mismos folds para todos), midiendo calidad y velocidad.

In [ ]:
from pathlib import Path

from src.model_training import modelos_clasificacion, modelos_regresion
from src.model_evaluation import (
    evaluar_modelos,
    guardar_resultados,
    resumir_resultados,
    tabla_legible,
)

RESULTADOS = Path("../results/metrics")

### 8. Modelos de clasificación

| Modelo | Idea en una frase | Por qué lo incluimos |
|---|---|---|
| Línea base | Siempre responde la clase más frecuente | Es el piso que hay que superar |
| Regresión Logística | Combina las variables con pesos y calcula la probabilidad de cada clase | Simple y rápida, pero solo capta relaciones "rectas" |
| KNN (k = 5) | Mira los 5 objetos más parecidos y copia su clase | No supone ninguna forma de los datos; por eso escalamos las variables |
| Árbol de decisión | Hace una serie de preguntas ("¿largo > 3 m?") hasta decidir | Fácil de interpretar, pero tiende a memorizar los datos |
| Random Forest | 100 árboles que votan entre todos | Corrige la tendencia del árbol a memorizar; es robusto |
| Gradient Boosting | Árboles en secuencia: cada uno corrige los errores del anterior | Suele ser de lo más preciso en datos de tabla, y es rápido |

**Métricas que medimos** (todas con 5 folds):
- **Accuracy:** porcentaje de aciertos. Engaña cuando las clases están desparejas.
- **Balanced accuracy:** promedio del recall de cada clase; trata igual a todas.
- **F1 macro:** equilibrio entre precision y recall, promediado entre clases.
- **Precision macro:** de lo que el modelo dice que es de una clase, cuánto realmente lo es.
- **ROC AUC (OvR):** qué tan bien ordena las probabilidades (0,5 = azar, 1 = perfecto).
- **Recall por clase:** de todos los objetos reales de una clase, cuántos encuentra.
- **Tiempos:** segundos para entrenar y milisegundos para predecir 1.000 filas.

In [ ]:
modelos_clf = modelos_clasificacion()

print("Evaluando modelos de clasificación (5 folds agrupados por segmento)...")
detalle_clf = evaluar_modelos(modelos_clf, "clasificacion", X_clf_train, y_clf_train,
                              g_clf_train, cv_clf)
resumen_clf = resumir_resultados(detalle_clf)

guardar_resultados(detalle_clf.set_index(["modelo", "fold"]), RESULTADOS / "clasificacion_cv_detalle.csv")
guardar_resultados(resumen_clf, RESULTADOS / "clasificacion_cv_resumen.csv")

#### Resultados de clasificación (media ± desviación entre folds)

In [ ]:
tabla_legible(resumen_clf, ["accuracy", "balanced_accuracy", "f1_macro", "precision_macro", "roc_auc_ovr"])

In [ ]:
print("Recall por clase (qué porcentaje de cada clase encuentra el modelo):")
tabla_legible(resumen_clf, ["recall_vehiculo", "recall_senal", "recall_peaton", "recall_ciclista"])

In [ ]:
print("Velocidad:")
tabla_legible(resumen_clf, ["tiempo_entrenamiento_s", "tiempo_prediccion_ms_por_1000"], decimales=2)

**Hallazgo:** _(¿qué modelo es mejor en balanced accuracy y F1 macro? ¿cuál es el más rápido? ¿cómo le va a cada modelo con la clase Ciclista? ¿el mejor en accuracy es también el mejor en balanced accuracy?)._

### 9. Modelos de regresión (predecir la distancia en metros)

Los mismos tipos de modelo, ahora para predecir un número. La regresión lineal usa Ridge, que es una regresión lineal con una pequeña penalización a los pesos muy grandes (más estable).

**Métricas:**
- **R²:** qué parte de la variación de la distancia explica el modelo (0 = nada, 1 = todo).
- **RMSE:** error típico en metros; castiga más los errores grandes.
- **MAE:** error promedio en metros; es el más fácil de explicar.

In [ ]:
modelos_reg = modelos_regresion()

print("Evaluando modelos de regresión (5 folds agrupados por segmento)...")
detalle_reg = evaluar_modelos(modelos_reg, "regresion", X_reg_train, y_reg_train,
                              g_reg_train, cv_reg)
resumen_reg = resumir_resultados(detalle_reg)

guardar_resultados(detalle_reg.set_index(["modelo", "fold"]), RESULTADOS / "regresion_cv_detalle.csv")
guardar_resultados(resumen_reg, RESULTADOS / "regresion_cv_resumen.csv")

#### Resultados de regresión (media ± desviación entre folds)

In [ ]:
tabla_legible(resumen_reg, ["r2", "rmse", "mae"])

In [ ]:
print("Velocidad:")
tabla_legible(resumen_reg, ["tiempo_entrenamiento_s", "tiempo_prediccion_ms_por_1000"], decimales=2)

**Hallazgo:** _(¿qué modelo explica mejor la distancia (R²) y cuál se equivoca menos en metros (MAE)? ¿cuánto mejora respecto de la línea base? ¿vale la pena el modelo más lento?)._

## Decisiones de la Parte B

_(complétalo con tus palabras: por qué elegimos estos modelos, qué modelo ganó en cada tarea y por qué, qué compromiso hay entre calidad y velocidad, y qué se espera mejorar con la optimización de hiperparámetros)._